In [1]:
!pip install protobuf==4.23.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.5/304.5 kB 6.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.23.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 4.23.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 4.23.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorf

In [2]:
# load libraries

# for data import and manipulation
import json
import numpy as np
import random

# for dataset building
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

# for model building and tracking
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from tqdm import tqdm

# for evaluation
from sklearn.metrics import classification_report as sklearn_classification_report

In [3]:
with open("/kaggle/input/training-validation-stance/training_set.json", "r") as f:
    data = json.load(f)

In [4]:
class StanceNLIDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_len, label2id):
        self.dataset = []

        for item in raw_data:
            sentence = item["sentence"]
            target = item["group"]
            gold_stance = item["stance"]
            
            hypotheses = {
                "pos": f"The text is positive towards {target}.",
                "neg": f"The text is negative towards {target}.",
                "neutral": f"The text is neutral, or contains no stance, towards {target}."
            }

            for stance, hypothesis in hypotheses.items():
                label_text = "entailment" if stance == gold_stance else "not_entailment"

                encoding = tokenizer(
                    sentence,
                    hypothesis,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                    )
                self.dataset.append({
                    "gold_stance": gold_stance,
                    "input_ids": encoding["input_ids"].squeeze(0),
                    "attention_mask": encoding["attention_mask"].squeeze(0),
                    "label": label2id[label_text]
                    })
                

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def evaluate_nli_stance(model, data):
    model.eval()
    all_preds = []
    all_labels = []

    for item in data:
        sentence = item["sentence"]
        target = item["group"]
        gold_stance = item["stance"]
        
        hypotheses = {
            "pos": f"The text is positive towards {target}.",
            "neg": f"The text is negative towards {target}.",
            "neutral": f"The text is neutral, or contains no stance, towards {target}."
            }

        # Tokenize all 3 hypotheses as a batch
        inputs = tokenizer(
            [sentence]*3,
            list(hypotheses.values()),
            return_tensors="pt",
            padding=True,
            truncation=True
            )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            entail_probs = probs[:, 0].tolist() # 0 is the entailment index

        # choose hypothesis with highest entailment probability
        predicted_stance = list(hypotheses.keys())[entail_probs.index(max(entail_probs))]

        all_labels.append(gold_stance)
        all_preds.append(predicted_stance)

    return all_labels, all_preds

# NLI Model (Laurer, 2024)
## CV of Original Model

In [5]:
# set seeds to ensure reproducibility of all remaining tasks
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [9]:
# create list to store cv results in
num_folds = 5
metrics_original_data = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 5
lr = 2e-05
weight_decay = 0.01
batch_size = 16

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data]
for fold, (train_idx, val_idx) in enumerate(kf.split(data, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id
    
    # split the data
    train_fold_data = [data[i] for i in train_idx]
    val_fold_data = [data[i] for i in val_idx]

    # create actual training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_original_data.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)

with open("/kaggle/working/metrics_original_data.json", "w") as f:
    json.dump(metrics_original_data, f)

Epoch 1/5


Training: 100%|██████████| 433/433 [01:37<00:00,  4.44it/s, loss=0.0419]


Average training loss: 0.2956
Epoch 2/5


Training: 100%|██████████| 433/433 [01:36<00:00,  4.50it/s, loss=0.0194]  


Average training loss: 0.1256
Epoch 3/5


...

Average training loss: 0.0535
Epoch 4/5


Training: 100%|██████████| 433/433 [01:36<00:00,  4.49it/s, loss=0.000163]


Average training loss: 0.0293
Epoch 5/5


Training: 100%|██████████| 433/433 [01:36<00:00,  4.49it/s, loss=0.000116]


Average training loss: 0.0138


In [10]:
metrics_original_data

[{'negative': {'mean': 0.7475341574495771,
   'sd': 0.0919066200238365,
   'lower': 0.6669744530670048,
   'upper': 0.8280938618321494},
  'neutral': {'mean': 0.7963759614809327,
   'sd': 0.013457412701347891,
   'lower': 0.7845800191571514,
   'upper': 0.808171903804714},
  'positive': {'mean': 0.8925717896675088,
   'sd': 0.008929736092211241,
   'lower': 0.8847445308735683,
   'upper': 0.9003990484614494},
  'macro_avg': {'mean': 0.8121606361993395,
   'sd': 0.03470701217131123,
   'lower': 0.7817385987030414,
   'upper': 0.8425826736956377}}]

## CV of Oversampling of Negative Class Only

In [11]:
# create list to store cv results in
num_folds = 5
metrics_aug_only_negative = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 5
lr = 2e-05
weight_decay = 0.01
batch_size = 16

# set the oversample proportion
oversample_prop = 0.5

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data]
for fold, (train_idx, val_idx) in enumerate(kf.split(data, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id
    
    # split the data
    train_fold_data = [data[i] for i in train_idx]
    val_fold_data = [data[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample the minority class
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*oversample_prop), random_state=0)
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=batch_size, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_only_negative.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)

with open("/kaggle/working/metrics_aug_only_negative.json", "w") as f:
    json.dump(metrics_aug_only_negative, f)

Epoch 1/5


Training: 100%|██████████| 444/444 [01:39<00:00,  4.44it/s, loss=0.146] 


Average training loss: 0.3057
Epoch 2/5


Training: 100%|██████████| 444/444 [01:39<00:00,  4.47it/s, loss=0.0204]  


Average training loss: 0.1182
Epoch 3/5


...

Average training loss: 0.0601
Epoch 4/5


Training: 100%|██████████| 444/444 [01:39<00:00,  4.48it/s, loss=0.000126]


Average training loss: 0.0247
Epoch 5/5


Training: 100%|██████████| 444/444 [01:38<00:00,  4.49it/s, loss=0.000187]


Average training loss: 0.0233


In [13]:
metrics_aug_only_negative

[{'negative': {'mean': 0.7662478215451465,
   'sd': 0.05290623980661733,
   'lower': 0.7198734576776779,
   'upper': 0.8126221854126151},
  'neutral': {'mean': 0.792181725945666,
   'sd': 0.024598797637886388,
   'lower': 0.7706199291419006,
   'upper': 0.8137435227494315},
  'positive': {'mean': 0.8898464068455508,
   'sd': 0.009083563022131074,
   'lower': 0.8818843128025585,
   'upper': 0.8978085008885431},
  'macro_avg': {'mean': 0.8160919847787877,
   'sd': 0.022960845243544567,
   'lower': 0.7959659165509038,
   'upper': 0.8362180530066717}}]

## CV of Oversampling Proportionally

In [12]:
# create list to store cv results in
num_folds = 5
metrics_aug_prop = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 5
lr = 2e-05
weight_decay = 0.01
batch_size = 16

# set the oversample proportion
oversample_prop = 0.25

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data]
for fold, (train_idx, val_idx) in enumerate(kf.split(data, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id
 
    # split the data
    train_fold_data = [data[i] for i in train_idx]
    val_fold_data = [data[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample all classes to a certain proportion and creare datasets
    pos_ann = [r for r in train_fold_data if r[0]["stance"] == "pos"]
    neu_ann = [r for r in train_fold_data if r[0]["stance"] == "neutral"]
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    pos_extra = resample(pos_ann, replace=False, n_samples=int(len(pos_ann)*oversample_prop), random_state=0)
    neu_extra = resample(neu_ann, replace=False, n_samples=int(len(neu_ann)*oversample_prop), random_state=0)
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*oversample_prop), random_state=0)
    train_data_pos_aug = [task[2] for task in pos_extra]
    train_data_neu_aug = [task[2] for task in neu_extra]
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_pos_aug = StanceNLIDataset(train_data_pos_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neu_aug = StanceNLIDataset(train_data_neu_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_pos_aug, train_dataset_neu_aug, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=batch_size, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_prop.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)
with open("/kaggle/working/metrics_aug_prop.json", "w") as f:
    json.dump(metrics_aug_prop, f)

Epoch 1/5


Training: 100%|██████████| 541/541 [02:01<00:00,  4.47it/s, loss=0.00566]


Average training loss: 0.2751
Epoch 2/5


Training: 100%|██████████| 541/541 [02:00<00:00,  4.48it/s, loss=0.00022] 


Average training loss: 0.0987
Epoch 3/5


...

Average training loss: 0.0393
Epoch 4/5


Training: 100%|██████████| 541/541 [01:59<00:00,  4.52it/s, loss=0.00011] 


Average training loss: 0.0240
Epoch 5/5


Training: 100%|██████████| 541/541 [01:59<00:00,  4.53it/s, loss=0.00038] 


Average training loss: 0.0176


In [14]:
metrics_aug_prop

[{'negative': {'mean': 0.7302647517709877,
   'sd': 0.0895208385527679,
   'lower': 0.6517962770515328,
   'upper': 0.8087332264904427},
  'neutral': {'mean': 0.8000509325455717,
   'sd': 0.021254298367361344,
   'lower': 0.7814207186078864,
   'upper': 0.818681146483257},
  'positive': {'mean': 0.8883242020828808,
   'sd': 0.01161239451731054,
   'lower': 0.8781454895021579,
   'upper': 0.8985029146636037},
  'macro_avg': {'mean': 0.80621329546648,
   'sd': 0.02783880493419249,
   'lower': 0.7818115070503573,
   'upper': 0.8306150838826027}}]

## CV of Oversampling of All Minority Classes

In [15]:
# create list to store cv results in
num_folds = 5
metrics_aug_negative_neutral = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 5
lr = 2e-05
weight_decay = 0.01
batch_size = 16

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data]
for fold, (train_idx, val_idx) in enumerate(kf.split(data, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id

    # split the data
    train_fold_data = [data[i] for i in train_idx]
    val_fold_data = [data[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample all classes to a certain proportion and create datasets
    neu_ann = [r for r in train_fold_data if r[0]["stance"] == "neutral"]
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    neu_extra = resample(neu_ann, replace=False, n_samples=int(len(neu_ann)*0.25), random_state=0)
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*0.5), random_state=0)
    train_data_neu_aug = [task[2] for task in neu_extra]
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_neu_aug = StanceNLIDataset(train_data_neu_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_neu_aug, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=batch_size, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_negative_neutral.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)
with open("/kaggle/working/metrics_negative_neutral.json", "w") as f:
    json.dump(metrics_aug_negative_neutral, f)

Epoch 1/5


Training: 100%|██████████| 483/483 [01:48<00:00,  4.47it/s, loss=0.0638] 


Average training loss: 0.2852
Epoch 2/5


Training: 100%|██████████| 483/483 [01:47<00:00,  4.50it/s, loss=0.00107] 


Average training loss: 0.1010
Epoch 3/5


...

Average training loss: 0.0440
Epoch 4/5


Training: 100%|██████████| 483/483 [01:46<00:00,  4.51it/s, loss=0.000106]


Average training loss: 0.0207
Epoch 5/5


Training: 100%|██████████| 483/483 [01:46<00:00,  4.52it/s, loss=0.00107] 


Average training loss: 0.0206


In [16]:
metrics_aug_negative_neutral

[{'negative': {'mean': 0.7146308488738938,
   'sd': 0.1150165190342263,
   'lower': 0.6138144248762899,
   'upper': 0.8154472728714978},
  'neutral': {'mean': 0.7851636322729616,
   'sd': 0.028724744067319726,
   'lower': 0.7599852839676038,
   'upper': 0.8103419805783194},
  'positive': {'mean': 0.8806255501592244,
   'sd': 0.009983782131284457,
   'lower': 0.8718743792761293,
   'upper': 0.8893767210423195},
  'macro_avg': {'mean': 0.7934733437686933,
   'sd': 0.047487121650495324,
   'lower': 0.7518490463987,
   'upper': 0.8350976411386866}}]